# LR-ASD Per-Interval Orchestrator

This notebook replaces the older GNU Parallel batch flow with a per-interval SLURM orchestration pattern.

For a chosen patient and camera serial, it:

1. discovers synced interval videos under `vad_new/{patient}/vad_data`
2. optionally filters to syncs marked `GOOD`
3. checks LR-ASD `_SUCCESS` and error markers per interval
4. submits one SLURM job per interval video
5. lets you inspect status/errors without relying only on `pyavi/video_out.avi`

Worker script:
[notebooks/REBIRTH_2!/video_processing/lrasd_interval_worker.py](/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/video_processing/lrasd_interval_worker.py)


In [1]:
from pathlib import Path
import subprocess
import re

import pandas as pd


In [2]:
# --------------------
# User-facing settings
# --------------------
PROJ_DIR = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247')
VAD_ROOT = PROJ_DIR / 'vad_new'

PATIENT = 'YFG'
CAMERA_SERIAL = '18486638'   # e.g. '23512014', '23512013', '18486634'
FILTER_SYNC_GOOD_ONLY = True
FORCE_RESUBMIT = False
MAX_INTERVALS = None         # e.g. 20 for a test run

WORKER_PATH = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/video_processing/lrasd_interval_worker.py')
PYTHON_BIN = '/scratch/tahaismail424/miniforge3/envs/asd_testing/bin/python3'
LRASD_REPO_ROOT = Path('/scratch/tahaismail424/asd_testing/LR-ASD')
MODEL_PATH = LRASD_REPO_ROOT / 'weight' / 'finetuning_TalkSet.model'
CONDA_ACTIVATE = 'source /scratch/tahaismail424/.bash_profile && conda activate asd_testing'

# SLURM settings
PARTITION = 'Universal'
GRES = 'mps:l40:16'
CPUS = 8
MEM = '64G'
TIME = '24:00:00'
QOS = 'big_batch_tier'                   # e.g. 'default_tier'

RUN_NAME = f'lrasd_{PATIENT}_{CAMERA_SERIAL}'
LOGS_DIR = VAD_ROOT / PATIENT / 'video_processing_slurm_logs' / CAMERA_SERIAL
SCRIPTS_DIR = VAD_ROOT / PATIENT / 'video_processing_slurm_scripts' / CAMERA_SERIAL
LOGS_DIR.mkdir(parents=True, exist_ok=True)
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

print('patient:     ', PATIENT)
print('camera:      ', CAMERA_SERIAL)
print('vad root:    ', VAD_ROOT / PATIENT)
print('worker:      ', WORKER_PATH)
print('repo root:   ', LRASD_REPO_ROOT)
print('model path:  ', MODEL_PATH)
print('logs dir:    ', LOGS_DIR)
print('scripts dir: ', SCRIPTS_DIR)


patient:      YFG
camera:       18486638
vad root:     /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFG
worker:       /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/video_processing/lrasd_interval_worker.py
repo root:    /scratch/tahaismail424/asd_testing/LR-ASD
model path:   /scratch/tahaismail424/asd_testing/LR-ASD/weight/finetuning_TalkSet.model
logs dir:     /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFG/video_processing_slurm_logs/18486638
scripts dir:  /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFG/video_processing_slurm_scripts/18486638


In [3]:
def find_sync_quality(mp4_path: Path) -> Path | None:
    sync_quality = mp4_path.parent / 'sync_quality.txt'
    return sync_quality if sync_quality.exists() else None


def read_sync_status(sync_quality_path: Path | None) -> str | None:
    if sync_quality_path is None:
        return None
    text = sync_quality_path.read_text(errors='ignore')
    match = re.search(r'STATUS:\s*(GOOD|BAD)', text)
    return match.group(1) if match else None


def derive_output_dir(mp4_path: Path) -> Path:
    return mp4_path.parent / mp4_path.stem


def derive_status_row(mp4_path: Path) -> dict:
    interval_id = mp4_path.parents[5].name
    output_dir = derive_output_dir(mp4_path)
    pyavi_dir = output_dir / 'pyavi'
    success_path = pyavi_dir / '_SUCCESS'
    error_path = pyavi_dir / 'asd_error.txt'
    video_out_path = pyavi_dir / 'video_out.avi'
    sync_quality_path = find_sync_quality(mp4_path)
    sync_status = read_sync_status(sync_quality_path)
    return {
        'interval_id': interval_id,
        'mp4_path': mp4_path,
        'output_dir': output_dir,
        'pyavi_dir': pyavi_dir,
        'success_path': success_path,
        'error_path': error_path,
        'video_out_path': video_out_path,
        'sync_quality_path': sync_quality_path,
        'sync_status': sync_status,
        'has_success': success_path.exists(),
        'has_error': error_path.exists(),
        'has_video_out': video_out_path.exists(),
    }


def build_status_df(patient: str, camera_serial: str) -> pd.DataFrame:
    patient_root = VAD_ROOT / patient / 'vad_data'
    pattern = f'*/video/*/neural/{camera_serial}/synced_video/neural_{camera_serial}.mp4'
    mp4_paths = sorted(patient_root.glob(pattern))

    rows = [derive_status_row(path) for path in mp4_paths]
    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df['sync_good'] = df['sync_status'].eq('GOOD')
    df['ready_for_submit'] = ~df['has_success']
    if FILTER_SYNC_GOOD_ONLY:
        df['ready_for_submit'] = df['ready_for_submit'] & df['sync_good']
    if FORCE_RESUBMIT:
        df['ready_for_submit'] = True
        if FILTER_SYNC_GOOD_ONLY:
            df['ready_for_submit'] = df['ready_for_submit'] & df['sync_good']

    df = df.sort_values('interval_id').reset_index(drop=True)
    if MAX_INTERVALS is not None:
        df = df.head(MAX_INTERVALS).copy()
    return df


In [4]:
status_df = build_status_df(PATIENT, CAMERA_SERIAL)
print('interval videos found:', len(status_df))
if not status_df.empty:
    print('sync GOOD:', int(status_df['sync_good'].sum()))
    print('success:', int(status_df['has_success'].sum()))
    print('errors:', int(status_df['has_error'].sum()))
    print('video_out exists:', int(status_df['has_video_out'].sum()))
    print('ready_for_submit:', int(status_df['ready_for_submit'].sum()))

status_df[[
    'interval_id', 'sync_status', 'has_success', 'has_error', 'has_video_out', 'ready_for_submit', 'mp4_path'
]] if not status_df.empty else pd.DataFrame()


interval videos found: 416
sync GOOD: 409
success: 0
errors: 0
video_out exists: 0
ready_for_submit: 409


,interval_id,sync_status,has_success,has_error,has_video_out,ready_for_submit,mp4_path
0,20240923-200755_0049,GOOD,False,False,False,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
1,20240923-200755_0050,GOOD,False,False,False,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
2,20240924-110019_0000,GOOD,False,False,False,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
3,20240924-110019_0001,GOOD,False,False,False,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
4,20240924-110019_0002,GOOD,False,False,False,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
...,...,...,...,...,...,...,...
411,20240927-145227_0005,GOOD,False,False,False,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
412,20240927-145227_0006,GOOD,False,False,False,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
413,20240927-145227_0007,GOOD,False,False,False,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
414,20240927-145227_0008,GOOD,False,False,False,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...


In [5]:
submitted = []
failed = []

if status_df.empty:
    print('No interval mp4 files found for this patient/camera serial.')
else:
    
    for _, row in status_df.iterrows():
        if not row['ready_for_submit']:
            continue

        mp4_path = row['mp4_path']
        output_dir = row['output_dir']
        success_path = row['success_path']
        error_path = row['error_path']

        reset_line = f'rm -f {success_path} {error_path}' if FORCE_RESUBMIT else 'true'
        qos_lines = [f'#SBATCH --qos={QOS}'] if QOS else []

        lines = [
            '#!/bin/bash',
            f'#SBATCH --job-name=asd_{row["interval_id"]}',
            f'#SBATCH --partition={PARTITION}',
            f'#SBATCH --gres={GRES}',
            f'#SBATCH --exclude=guppy-1',
            f'#SBATCH --cpus-per-task={CPUS}',
            f'#SBATCH --mem={MEM}',
            f'#SBATCH --time={TIME}',
            *qos_lines,
            f'#SBATCH --output={LOGS_DIR}/{row["interval_id"]}_%j.out',
            f'#SBATCH --error={LOGS_DIR}/{row["interval_id"]}_%j.err',
            '',
            'set -eo pipefail',
            '',
            CONDA_ACTIVATE,
            '',
            'echo "HOSTNAME: $(hostname)"',
            'echo "START: $(date)"',
            f'echo "PATIENT: {PATIENT}"',
            f'echo "CAMERA_SERIAL: {CAMERA_SERIAL}"',
            f'echo "INTERVAL_ID: {row["interval_id"]}"',
            f'echo "VIDEO_MP4: {mp4_path}"',
            '',
            reset_line,
            '',
            f'{PYTHON_BIN} {WORKER_PATH} {mp4_path} {LRASD_REPO_ROOT} {MODEL_PATH}',
            '',
            'echo "END: $(date)"',
        ]
        sbatch_text = '\n'.join(lines) + '\n'

        sbatch_path = SCRIPTS_DIR / f'{row["interval_id"]}.sbatch'
        sbatch_path.write_text(sbatch_text)

        try:
            res = subprocess.run(['sbatch', str(sbatch_path)], capture_output=True, text=True, check=True)
            submitted.append((row['interval_id'], res.stdout.strip()))
            print(f'submitted: {row["interval_id"]} -> {res.stdout.strip()}')
        except subprocess.CalledProcessError as exc:
            failed.append((row['interval_id'], exc.stderr.strip()))
            print(f'FAILED SUBMISSION: {row["interval_id"]}')
            print(exc.stderr)

print(f'submitted={len(submitted)}, failed={len(failed)}')


submitted: 20240923-200755_0049 -> Submitted batch job 253826
submitted: 20240923-200755_0050 -> Submitted batch job 253827
submitted: 20240924-110019_0000 -> Submitted batch job 253828
submitted: 20240924-110019_0001 -> Submitted batch job 253829
submitted: 20240924-110019_0002 -> Submitted batch job 253830
submitted: 20240924-110019_0003 -> Submitted batch job 253831
submitted: 20240924-110019_0004 -> Submitted batch job 253832
submitted: 20240924-123425_0000 -> Submitted batch job 253833
submitted: 20240924-123425_0001 -> Submitted batch job 253834
submitted: 20240924-123425_0002 -> Submitted batch job 253835
submitted: 20240924-123425_0003 -> Submitted batch job 253836
submitted: 20240924-123425_0004 -> Submitted batch job 253837
submitted: 20240924-123425_0005 -> Submitted batch job 253838
submitted: 20240924-123425_0006 -> Submitted batch job 253839
submitted: 20240924-123425_0007 -> Submitted batch job 253840
submitted: 20240924-131507_0000 -> Submitted batch job 253841
submitte

In [ ]:
# Refresh status after jobs run.
status_df = build_status_df(PATIENT, CAMERA_SERIAL)
status_df[[
    'interval_id', 'sync_status', 'has_success', 'has_error', 'has_video_out', 'ready_for_submit'
]] if not status_df.empty else pd.DataFrame()


In [ ]:
# Inspect a few failures quickly.
if status_df.empty:
    print('No intervals found.')
else:
    error_rows = status_df[status_df['has_error']].copy()
    print(f'intervals with error files: {len(error_rows)}')
    for _, row in error_rows.head(5).iterrows():
        print('=' * 100)
        print(row['interval_id'])
        print(row['error_path'])
        print(row['error_path'].read_text()[:4000])


In [ ]:
# Optional: show intervals that produced video_out.avi but still do not have _SUCCESS.
if status_df.empty:
    print('No intervals found.')
else:
    suspicious = status_df[(status_df['has_video_out']) & (~status_df['has_success'])]
    suspicious[['interval_id', 'mp4_path', 'video_out_path', 'error_path']] if not suspicious.empty else pd.DataFrame()
